# Football Market Value Predictor

### 1. Loading Libraries & Datasets

In [1]:
import pandas as pd
players = pd.read_csv("data/players.csv.gz") # Player data
appearances = pd.read_csv("data/appearances.csv.gz") # Appearance data for each player
valuations = pd.read_csv("data/player_valuations.csv.gz") # Market Valuation 

#### Helper Function

In [2]:
def format_season(year):
    return (
        year.astype(str)
        + "/"
        + ((year + 1) % 100).astype(str).str.zfill(2))

### 2. Preparing Player Information

In [3]:
# Filtering columns that are useful for the model
player_cols = [
    "player_id",
    "name",
    "date_of_birth",
    "position",
    "sub_position",
    "foot",
    "height_in_cm",
    "country_of_citizenship"]

player_info = players[player_cols].copy()

In [4]:
# Converting string to time
appearances["date"] = pd.to_datetime(appearances["date"])
appearances["season_start"] = appearances["date"].dt.year

# For January through June matches, use the previous calendar year.
appearances.loc[
    appearances["date"].dt.month <= 6,
    "season_start"] -= 1

#2024 -> 2024/25
appearances["season"] = (
    format_season(appearances["season_start"]))

- `% 100` extracts the last two digits of a year, such as `25` from `2025`.
- `.str.zfill(2)` keeps a leading zero for years such as `2009`, producing `09`.

### 3. Defining Top-7 League Eligibility

In [5]:
top7_leagues = [
    "GB1",  # Premier League
    "ES1",  # La Liga
    "IT1",  # Serie A
    "L1",   # Bundesliga
    "FR1",  # Ligue 1
    "PO1",  # Primeira Liga
    "NL1"   # Eredivisie
    ]

In [6]:
# Only selects players that are from top 7 leagues as defined above, from the last 10 years
league_filter = appearances["competition_id"].isin(top7_leagues)
season_filter = appearances["season_start"].between(2016, 2025)
eligibility_filter = league_filter & season_filter

filtered_appearances = appearances[eligibility_filter].copy()
eligible_player_seasons = filtered_appearances[["player_id", "season"]].drop_duplicates()

### 4. Building All-Comp Player Statistics

In [7]:
# Selects appearances for players that are eligible
all_comp_data = pd.merge(
    appearances,
    eligible_player_seasons,
    on=["player_id", "season"],
    how="inner")

In [8]:
# Combine match appearances into all-competition season totals
player_season_stats = (
    all_comp_data.groupby(["player_id", "player_name", "season"])
    .agg(
        appearances=("game_id", "nunique"),
        goals=("goals", "sum"),
        assists=("assists", "sum"),
        minutes=("minutes_played", "sum")
    )
    .reset_index()
)

### 5. Adding League & Player Profile Information

In [9]:
# Converting league name codes to readable names
league_names = {
    "GB1": "Premier League",
    "ES1": "La Liga",
    "IT1": "Serie A",
    "L1": "Bundesliga",
    "FR1": "Ligue 1",
    "PO1": "Primeira Liga",
    "NL1": "Eredivisie"
}

filtered_appearances["league"] = (filtered_appearances["competition_id"].map(league_names))
# map looks for each comp id, and replaces it to assigned values from the referenced dict above

In [10]:
filtered_appearances = filtered_appearances.sort_values("date")
player_leagues = (
    filtered_appearances
    .groupby(["player_id", "season"])
    .tail(1))

player_leagues = player_leagues[["player_id", "season", "league"]].copy()

In [11]:
# Create player_performance by combining season totals, player info, and league data.
player_performance = (
    player_season_stats
    .merge(
        player_info.drop(columns=["name"]),
        on="player_id",
        how="left")
    .merge(
        player_leagues,
        on=["player_id", "season"],
        how="left"))

### 6. Calculating Player Age (as of August 1 that season)

In [12]:
player_performance["date_of_birth"] = pd.to_datetime(
    player_performance["date_of_birth"])

season_year = player_performance["season"].str[:4].astype(int)

player_performance["age"] = (season_year - player_performance["date_of_birth"].dt.year)

birthday_not_reached = (
    (player_performance["date_of_birth"].dt.month > 8)
    |
    ((player_performance["date_of_birth"].dt.month == 8)
        & (player_performance["date_of_birth"].dt.day > 1)
    )
)
player_performance["age"] = (player_performance["age"] - birthday_not_reached.astype(int))
player_performance = player_performance.drop(columns=["date_of_birth"])

### 7. Preparing Historical Market Valuations

In [13]:
# Converting valuation dates into matching football seasons
valuations["date"] = pd.to_datetime(valuations["date"])
valuations["season_start"] = valuations["date"].dt.year

valuations.loc[valuations["date"].dt.month <= 6, "season_start"] -= 1

valuations["season"] = (
    format_season(valuations["season_start"]))

In [14]:
# Keep each player's latest market value from each season
filtered_valuations = valuations[valuations["season_start"].between(2015, 2025)].copy()
# Include 2015/16 valuations to provide previous-season values for 2016/17 for model V3
filtered_valuations = filtered_valuations.sort_values("date")
player_season_values = (filtered_valuations.groupby(["player_id", "season"]).tail(1))
player_season_values = player_season_values[["player_id","season", "market_value_in_eur"]].copy()

# Converting market value to "in millions"
player_season_values = player_season_values.rename(
    columns={"market_value_in_eur": "market_value"})
player_season_values["market_value"] = (player_season_values["market_value"] / 1000000)

### 8. Building the Final Modeling Dataset

In [15]:
# Keep players with known market values and at least 450 minutes.
model_data = pd.merge(
    player_performance,
    player_season_values,
    on=["player_id", "season"],
    how="left")

model_data = model_data.dropna(subset=["market_value", "age"]).copy()
model_data = model_data[model_data["minutes"] >= 450].copy()

### 9. Preparing Features and Target (V1)

In [16]:
feature_cols_V1 = [
    "age",
    "appearances",
    "goals",
    "assists",
    "minutes"]

X_V1 = model_data[feature_cols_V1].copy()
y = model_data["market_value"].copy()

### 10. Creating Time-Based Training and Test Sets

In [17]:
# Train on older seasons and test on the three newest seasons
train_seasons = [
    "2016/17",
    "2017/18",
    "2018/19",
    "2019/20",
    "2020/21",
    "2021/22",
    "2022/23"]

test_seasons = [
    "2023/24",
    "2024/25",
    "2025/26"]

train_filter = model_data["season"].isin(train_seasons)
test_filter = model_data["season"].isin(test_seasons)

X_train_V1 = X_V1[train_filter]
X_test_V1 = X_V1[test_filter]

y_train = y[train_filter]
y_test = y[test_filter]

### 11. Training the Baseline Linear Regression Model (V1)

In [18]:
from sklearn.linear_model import LinearRegression

model_V1 = LinearRegression()

model_V1.fit(X_train_V1, y_train)

y_pred_V1 = model_V1.predict(X_test_V1)

In [19]:
# Evaluate V1 against a simple median-value baseline
from sklearn.metrics import mean_absolute_error

MAE_V1 = mean_absolute_error(y_test, y_pred_V1)

baseline_pred = pd.Series(
    y_train.median(),
    index=y_test.index
)

baseline_MAE = mean_absolute_error(
    y_test,
    baseline_pred
)

print(f"V1 Linear Regression MAE: €{MAE_V1:.2f}M")
print(f"Median Baseline MAE: €{baseline_MAE:.2f}M")

V1 Linear Regression MAE: €8.16M
Median Baseline MAE: €9.80M


### V1 Results

| Metric | Result |
|---|---:|
| Median baseline MAE | €9.80M |
| V1 MAE | €8.16M |
| Improvement over baseline | 16.7% |

V1 used age and season performance with Linear Regression. Its main limitation was producing unrealistic negative values for some players.

### Model V2: Log-Transformed Target

In [20]:
# V2 trains on log market value instead of raw market value
import numpy as np

y_train_log = np.log(y_train)

model_V2 = LinearRegression()

model_V2.fit(X_train_V1, y_train_log)

y_pred_log_V2 = model_V2.predict(X_test_V1)

# Convert predictions back to € millions
y_pred_V2 = np.exp(y_pred_log_V2)

MAE_V2 = mean_absolute_error(
    y_test,
    y_pred_V2
)

print(f"V2 MAE: €{MAE_V2:.2f}M")

V2 MAE: €7.53M


### V2 Results

| Metric | Result |
|---|---:|
| V1 MAE | €8.16M |
| V2 MAE | €7.53M |
| Improvement over V1 | 7.7% |
| Improvement over baseline | 23.2% |

V2 used the same features as V1 but predicted log market value. This improved accuracy and prevented negative predictions.

### Model V3: Adding Previous Market Value

In [21]:
value_history = player_season_values.copy()
value_history["season_start"] = (
    value_history["season"].str[:4].astype(int))

previous_values = value_history[["player_id", "season_start", "market_value"]].copy()
previous_values = previous_values.rename(
    columns={
        "season_start": "previous_season", 
        "market_value": "previous_market_value"})

value_history["previous_season"] = (value_history["season_start"] - 1)

value_history = pd.merge(
    value_history.drop(columns=["season_start"]),
    previous_values,
    on=["player_id", "previous_season"],
    how="left"
)
value_history["previous_season"] = format_season(
    value_history["previous_season"])

In [22]:
model_data_V3 = pd.merge(
    model_data,
    value_history[
        ["player_id", "season", "previous_market_value"]
    ],
    on=["player_id", "season"],
    how="left"
)
# V3 requires a known previous-season market value
model_data_V3 = model_data_V3.dropna(
    subset=["previous_market_value"]).copy()

model_data_V3["previous_log_market_value"] = np.log(
    model_data_V3["previous_market_value"])


In [23]:
# V3 adds previous market value to the V1/V2 features
feature_cols_V3 = feature_cols_V1 + ["previous_log_market_value"]

X_V3 = model_data_V3[feature_cols_V3].copy()

y_V3 = model_data_V3["market_value"].copy()

train_filter_V3 = model_data_V3["season"].isin(train_seasons)
test_filter_V3 = model_data_V3["season"].isin(test_seasons)

X_train_V3 = X_V3[train_filter_V3]
X_test_V3 = X_V3[test_filter_V3]

y_train_V3 = y_V3[train_filter_V3]
y_test_V3 = y_V3[test_filter_V3]

y_train_log_V3 = np.log(y_train_V3)

In [24]:
model_V3 = LinearRegression()

model_V3.fit(
    X_train_V3,
    y_train_log_V3
)

y_pred_log_V3 = model_V3.predict(X_test_V3)

# Convert predictions back to € millions
y_pred_V3 = np.exp(y_pred_log_V3)

MAE_V3 = mean_absolute_error(
    y_test_V3,
    y_pred_V3
)

print(f"V3 MAE: €{MAE_V3:.2f}M")

V3 MAE: €3.72M


In [25]:
# Compare V3 against simply using last season's value
previous_value_pred = model_data_V3.loc[
    test_filter_V3,
    "previous_market_value"]

previous_value_MAE = mean_absolute_error(
    y_test_V3,
    previous_value_pred)

print(f"Previous Value Baseline MAE: €{previous_value_MAE:.2f}M")

Previous Value Baseline MAE: €4.26M


### V3 Results

| Metric | Result |
|---|---:|
| V2 MAE on the same test rows | €7.58M |
| Previous-value baseline MAE | €4.26M |
| V3 MAE | €3.72M |
| Improvement over previous-value baseline | 12.7% |

V3 added the player's previous-season market value, which produced the largest improvement so far. It could still overestimate superstars.

### Model V4: Random Forest with V3 Features

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model_V4 = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
)

model_V4.fit(
    X_train_V3,
    y_train_V3
)
y_pred_V4 = model_V4.predict(X_test_V3)

MAE_V4 = mean_absolute_error(
    y_test_V3,
    y_pred_V4
)

print(f"V4 Random Forest MAE: €{MAE_V4:.2f}M")

V4 Random Forest MAE: €3.29M


### V4 Results

| Metric | Result |
|---|---:|
| Model | Random Forest Regressor |
| Features | V3 features |
| V4 MAE | €3.29M |

V4 improved on V3 by modeling nonlinear relationships, but it still struggles with the most expensive players because important market signals are missing.

### Project Summary: Football Market Value Prediction

**Goal:** Predict player market value in € millions using season performance and historical market value.

**Evaluation:** Training seasons: 2016/17 to 2022/23. Test seasons: 2023/24 to 2025/26.

| Model | Main change | MAE | Change | Issue or lesson |
|---|---|---:|---:|---|
| V1 | Raw-target Linear Regression | €8.16M | Baseline | Negative predictions for some players. |
| V2 | Log-transformed target | €7.53M | 7.7% lower than V1 | Positive predictions, but limited context. |
| V3 | Previous-season market value | €3.72M | 50.9% lower on V3-compatible rows | Superstar overestimation remained. |
| V4 | Random Forest with V3 features | **€3.29M** | 11.6% lower than V3 | Weak extrapolation for the most expensive players. |

**V1 to V4:** On the same 8,255 V3-compatible test rows, MAE decreased from €8.19M to €3.29M, a 59.9% reduction. The original V1 result was €8.16M on the larger V1 cohort.

**Main takeaway:** Historical market value produced the largest improvement. The remaining error is concentrated among superstar players, where important market signals are missing from the dataset.

**Limitation:** Superstar values remain harder to predict because the dataset does not include factors such as contracts, wages, reputation, and transfer demand.

**Resume summary:** Built a time-aware football market-value prediction pipeline from raw player, appearance, and valuation data. Reduced MAE to €3.29M using historical market value and Random Forest regression, then analyzed errors among high-value players.